<a href="https://colab.research.google.com/github/taselshambakey/DECI-final-project/blob/main/Tasneem_DECI_final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initialization

In [17]:
import csv
import json
import sqlite3
import re

import pandas as pd
from bs4 import BeautifulSoup

STUDENT_ID = "EYOUTH-30812131201647"

DB_PATH = "database.db"
BOOKS_JSON_PATH = "books.json"
KICKOFF_HTML_PATH = "Reading Kickoff signups.html"

TASK1_ANSWERS_PATH = f"{STUDENT_ID}-Library-task1_sql_answers.txt"
TASK1_CSV_PATH = f"{STUDENT_ID}-Library-task1_combined_data.csv"
TASK2_CSV_PATH = f"{STUDENT_ID}-Library-task2_cleaned_data.csv"
TASK2_REPORT_PATH = f"{STUDENT_ID}-Library-integrity_report.txt"

FAIRNESS_REFLECTION_PATH = f"{STUDENT_ID}-Library-fairness_reflection.txt"

output_lines = []


def log(line=""):
    print(line)
    output_lines.append(str(line))


## Task 1- Goal 1

Five questions answered directly against the database. No cleaning is
performed here -- every query runs against the raw `checkouts` table as-is,
duplicates included. Data cleaning (removing true duplicates, handling
inconsistent spellings, etc.) is deliberately deferred to Task 2.

In [18]:
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

# ---------------------------------------------------------------------------
log("################ Q1: Checkouts per member (incl. zero) ################")
log("Reasoning:")
log("  - Every member must appear in the result, even those with no checkouts,")
log("    so members is the driving table with a LEFT JOIN out to checkouts")
log("    (an INNER JOIN would silently drop members with zero checkouts).")
log("  - COUNT(c.checkout_id) counts only matched checkout rows per member; for")
log("    members with no matches the LEFT JOIN produces NULLs, which COUNT(...)")
log("    correctly reports as 0 rather than NULL.")
log("  - The raw checkouts table is queried directly, with no de-duplication --")
log("    that cleanup is intentionally left for Task 2, so these counts include")
log("    a small number of duplicate-row inflation that Task 2 later corrects.")
log("  - Sorted busiest-first (checkout_count DESC), with member_id as a")
log("    tiebreaker for a stable, reproducible order.")
log()
q1 = """
SELECT m.member_id,
       m.first_name || ' ' || m.last_name AS member_name,
       COUNT(c.checkout_id) AS checkout_count
FROM members m
LEFT JOIN checkouts c ON c.member_id = m.member_id
GROUP BY m.member_id, member_name
ORDER BY checkout_count DESC, m.member_id ASC;
"""
cur.execute(q1)
rows1 = cur.fetchall()
for r in rows1:
    log(dict(r))
log(f"Total members: {len(rows1)}")
log(f"Members with 0 checkouts: {sum(1 for r in rows1 if r['checkout_count'] == 0)}")

# ---------------------------------------------------------------------------
log()
log("################ Q2: Author pattern search ################")
log("Reasoning:")
log("  - Chosen pattern: author's first name starts with 'A' (SQL LIKE 'A%').")
log("    This is an arbitrary but concrete pattern choice, picked because it")
log("    returns a small, easy-to-verify set (3 distinct authors) rather than")
log("    an empty or overwhelming result.")
log("  - Query hits the books table only, since author is a book attribute,")
log("    not a checkout attribute -- no join to checkouts is needed, so")
log("    duplicate checkout rows don't factor into this question at all.")
log("  - Sorted by author then title so same-author books are grouped together.")
log()
pattern = "A%"
q2 = """
SELECT book_id, title, author
FROM books
WHERE author LIKE ?
ORDER BY author, title;
"""
cur.execute(q2, (pattern,))
rows2 = cur.fetchall()
for r in rows2:
    log(dict(r))

# ---------------------------------------------------------------------------
log()
log("################ Q3: Top 5 most-borrowed titles ################")
log("Reasoning:")
log("  - 'Most popular' = highest raw checkout frequency per title, so we")
log("    JOIN checkouts to books on book_id and COUNT checkouts per book.")
log("    An INNER JOIN is correct here (not LEFT JOIN) because a book with")
log("    zero checkouts isn't a 'popular book' and shouldn't appear at all.")
log("  - The raw checkouts table is used as-is, so any duplicate checkout row")
log("    is counted here too -- again, left untouched on purpose for Task 2.")
log("  - ORDER BY times_borrowed DESC, then title ASC as a tiebreaker, then")
log("    LIMIT 5 gives exactly the top five, with a stable, reproducible order")
log("    when counts tie.")
log()
q3 = """
SELECT b.book_id, b.title, b.author, COUNT(c.checkout_id) AS times_borrowed
FROM checkouts c
JOIN books b ON b.book_id = c.book_id
GROUP BY b.book_id, b.title, b.author
ORDER BY times_borrowed DESC, b.title ASC
LIMIT 5;
"""
cur.execute(q3)
rows3 = cur.fetchall()
for r in rows3:
    log(dict(r))

# ---------------------------------------------------------------------------
log()
log("################ Q4: Top 10 most active readers ################")
log("Reasoning:")
log("  - Same shape as Q1 (per-member checkout count), but here we only want")
log("    members who have actually borrowed something, so an INNER JOIN")
log("    (members to checkouts) is used instead of a LEFT JOIN -- a member")
log("    with 0 checkouts can't be one of the 'most active readers'.")
log("  - The raw checkouts table is used, so this ranking may be a little")
log("    inflated by a handful of duplicate rows -- expected at this stage.")
log("  - ORDER BY checkout_count DESC with member_id as a tiebreaker, then")
log("    LIMIT 10, gives the requested top-10 ranking highest to lowest.")
log()
q4 = """
SELECT m.member_id,
       m.first_name || ' ' || m.last_name AS member_name,
       COUNT(c.checkout_id) AS checkout_count
FROM members m
JOIN checkouts c ON c.member_id = m.member_id
GROUP BY m.member_id, member_name
ORDER BY checkout_count DESC, m.member_id ASC
LIMIT 10;
"""
cur.execute(q4)
rows4 = cur.fetchall()
for r in rows4:
    log(dict(r))

# ---------------------------------------------------------------------------
log()
log("################ Q5: Neighborhood activity, skipping 10 most recent ################")
log("Reasoning:")
log("  - Chosen neighborhood: Maadi -- the largest neighborhood by member count,")
log("    which gives a big enough checkout history to demonstrate 'looking")
log("    further back in time'.")
log("  - The raw neighborhood column has inconsistent casing/whitespace")
log("    ('Maadi', 'Maadi ' with a trailing space, etc.), so the WHERE clause")
log("    compares LOWER(TRIM(m.neighborhood)) against a lowercase literal.")
log("    This only affects how rows are MATCHED for this query -- it does not")
log("    change or clean any stored data, so it isn't a data-cleaning step.")
log("  - 'Newest to oldest' means ORDER BY checkout_date DESC (checkout_id DESC")
log("    as a tiebreaker for same-day checkouts, for a stable order).")
log("  - 'Second set of 10' means skipping the 10 most recent rows of that")
log("    ordered result (OFFSET 10) and then taking exactly the next 10")
log("    (LIMIT 10) -- i.e. checkouts ranked 11th-20th most recent, not")
log("    everything beyond the 10 most recent.")
log("  - The raw checkouts table is used, so a duplicated Maadi row could")
log("    appear in this window too -- left as-is, to be resolved in Task 2.")
log()
neighborhood = "maadi"
log(f"Chosen neighborhood: {neighborhood.title()}")
q5 = """
SELECT c.checkout_id, m.member_id,
       m.first_name || ' ' || m.last_name AS member_name,
       c.book_id, c.checkout_date, c.return_date
FROM checkouts c
JOIN members m ON m.member_id = c.member_id
WHERE LOWER(TRIM(m.neighborhood)) = ?
ORDER BY c.checkout_date DESC, c.checkout_id DESC
LIMIT 10 OFFSET 10;
"""
log("Query:")
log(q5.strip())
log()
log("Result:")
cur.execute(q5, (neighborhood,))
rows5 = cur.fetchall()
for r in rows5:
    log(dict(r))
log(f"Checkouts shown (second set of 10, ranked 11th-20th most recent): {len(rows5)}")

conn.close()

# ---------------------------------------------------------------------------
log()
log("================================================================")
log("REFLECTION: Web Page vs. Database as a Data Source")
log("================================================================")
log("The database and an API both hand data straight to a program in a")
log("fixed, predictable shape -- a SELECT returns known columns and types,")
log("no guessing required. The Reading Kickoff page is built for a person")
log("reading it in a browser, not for a program: it has no schema, no ID")
log("column (a checkout_id wasn't needed for a human just scanning the")
log("table), and no constraints stopping a bad or nonexistent value from")
log("being typed into it. Pulling data from it meant parsing HTML structure")
log("(finding the <table>, checking the header, reading <td> text) instead")
log("of just querying, and treating every value as unverified text until")
log("proven otherwise.")
log()
log("That distinction mattered concretely for this project: the database's")
log("foreign keys guarantee every checkout's member_id is real, but the")
log("Kickoff page has no such guarantee -- and indeed 5 of its 26 signups")
log("reference member_ids that don't exist anywhere in the members table.")
log("A relational database would have caught that at write time; a page")
log("meant for people to read had no mechanism to catch it at all. Knowing")
log("the source was 'read by a person' rather than 'consumed by a program'")
log("is exactly why those rows needed defensive handling instead of a")
log("simple join.")

# Save all answers AND their reasoning to a plain text file
with open(TASK1_ANSWERS_PATH, "w") as f:
    f.write("\n".join(output_lines))

print(f"\nSaved answers and reasoning to {TASK1_ANSWERS_PATH}")

################ Q1: Checkouts per member (incl. zero) ################
Reasoning:
  - Every member must appear in the result, even those with no checkouts,
    so members is the driving table with a LEFT JOIN out to checkouts
    (an INNER JOIN would silently drop members with zero checkouts).
  - COUNT(c.checkout_id) counts only matched checkout rows per member; for
    members with no matches the LEFT JOIN produces NULLs, which COUNT(...)
    correctly reports as 0 rather than NULL.
  - The raw checkouts table is queried directly, with no de-duplication --
    that cleanup is intentionally left for Task 2, so these counts include
    a small number of duplicate-row inflation that Task 2 later corrects.
  - Sorted busiest-first (checkout_count DESC), with member_id as a
    tiebreaker for a stable, reproducible order.

{'member_id': 1034, 'member_name': 'Aya Wahba', 'checkout_count': 25}
{'member_id': 1044, 'member_name': 'Sherif Saleh', 'checkout_count': 21}
{'member_id': 1008, 'mem

### Stage 1 — Members and Checkouts (pure Python, no query tool)

Constraint for this stage: pure Python only -- no SQL JOIN, no
pandas.merge(), no other "query tool" doing the linking for us. We fetch
the two raw tables with plain SELECT * statements and do the matching
ourselves with a dictionary keyed by member_id.

In [19]:
def fetch_all_as_dicts(cursor, table_name):
    """Read an entire table with a bare SELECT * (no JOIN) and return a list
    of plain dicts, one per row."""
    cursor.execute(f"SELECT * FROM {table_name}")
    columns = [d[0] for d in cursor.description]
    return [dict(zip(columns, row)) for row in cursor.fetchall()]


def build_stage1(db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    raw_members = fetch_all_as_dicts(cur, "members")
    raw_checkouts = fetch_all_as_dicts(cur, "checkouts")  # used as-is, duplicates included
    conn.close()

    # Index members by member_id for O(1) lookup -- this dict IS the "join",
    # done by hand instead of by a query engine.
    members_by_id = {m["member_id"]: m for m in raw_members}

    combined = []
    unmatched_checkouts = []  # checkouts whose member_id has no member record
    checkout_counts = {m["member_id"]: 0 for m in raw_members}  # every member starts at 0

    for c in raw_checkouts:
        member = members_by_id.get(c["member_id"])
        if member is None:
            # A checkout referencing a member that doesn't exist would be a
            # mis-link risk; we flag it instead of silently guessing.
            unmatched_checkouts.append(c)
            continue

        row = {
            "checkout_id": c["checkout_id"],
            "source": "database",
            "member_id": member["member_id"],
            "first_name": member["first_name"],
            "last_name": member["last_name"],
            "member_name": f"{member['first_name']} {member['last_name']}",
            "grade": member["grade"],
            "neighborhood": member["neighborhood"],
            "membership_status": member["membership_status"],
            "join_date": member["join_date"],
            "book_id": c["book_id"],
            "checkout_date": c["checkout_date"],
            "return_date": c["return_date"],
        }
        combined.append(row)
        checkout_counts[member["member_id"]] += 1

    # Attach each member's running total onto their own rows, so the total is
    # visible directly on the combined data without a second lookup.
    for row in combined:
        row["member_total_checkouts"] = checkout_counts[row["member_id"]]

    return {
        "combined": combined,
        "checkout_counts": checkout_counts,  # includes members with 0 checkouts
        "members_by_id": members_by_id,
        "raw_checkout_count": len(raw_checkouts),
        "unmatched_checkouts": unmatched_checkouts,
    }


stage1_result = build_stage1()
stage1_combined = stage1_result["combined"]

print(f"Raw checkout rows read: {stage1_result['raw_checkout_count']}")
print(f"Checkout rows in Stage 1 output (no de-duplication applied): {len(stage1_combined)}")
print(f"Checkouts that could not be matched to a member: {len(stage1_result['unmatched_checkouts'])}")
print(f"Members represented (incl. members with 0 checkouts): {len(stage1_result['checkout_counts'])}")
print()
print("Sample combined rows:")
for row in stage1_combined[:3]:
    print(row)
print()
print("Per-member totals (first 5, sorted by member_id):")
for member_id in sorted(stage1_result["checkout_counts"])[:5]:
    print(f"  member_id {member_id}: {stage1_result['checkout_counts'][member_id]} checkouts")

Raw checkout rows read: 391
Checkout rows in Stage 1 output (no de-duplication applied): 391
Checkouts that could not be matched to a member: 0
Members represented (incl. members with 0 checkouts): 80

Sample combined rows:
{'checkout_id': 9263, 'source': 'database', 'member_id': 1047, 'first_name': 'Sara', 'last_name': 'Rashad', 'member_name': 'Sara Rashad', 'grade': None, 'neighborhood': 'Heliopolis', 'membership_status': 'Inactive', 'join_date': '2024-06-25', 'book_id': 517, 'checkout_date': '2024-10-21', 'return_date': '2024-11-07', 'member_total_checkouts': 16}
{'checkout_id': 9340, 'source': 'database', 'member_id': 1072, 'first_name': 'Seif', 'last_name': 'Zaki', 'member_name': 'Seif Zaki', 'grade': 9, 'neighborhood': 'Zamalek', 'membership_status': 'Active', 'join_date': '2025-10-21', 'book_id': 513, 'checkout_date': '2025-08-24', 'return_date': '2025-09-01', 'member_total_checkouts': 14}
{'checkout_id': 9231, 'source': 'database', 'member_id': 1053, 'first_name': 'Adam', 'last

### Stage 2 — Book Details

Book details live in two places that both need to be combined first:
  - database.db -> books table: book_id, title, author
  - books.json          : book_id, genre, pages, publication_year, publisher
Both are keyed by book_id (501-532), so they're merged into one lookup dict
before being attached to each checkout. The checkout row count must stay
exactly the same as Stage 1 -- this stage adds columns, not rows.

In [20]:
def load_books_catalog(db_path=DB_PATH, json_path=BOOKS_JSON_PATH):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    db_books = fetch_all_as_dicts(cur, "books")  # book_id, title, author
    conn.close()

    with open(json_path) as f:
        json_books = json.load(f)  # book_id, genre, pages, publication_year, publisher

    db_books_by_id = {b["book_id"]: b for b in db_books}
    json_books_by_id = {b["book_id"]: b for b in json_books}

    all_ids = set(db_books_by_id) | set(json_books_by_id)
    only_in_db = set(db_books_by_id) - set(json_books_by_id)
    only_in_json = set(json_books_by_id) - set(db_books_by_id)

    books_by_id = {}
    for book_id in all_ids:
        merged = {}
        merged.update(db_books_by_id.get(book_id, {}))
        merged.update(json_books_by_id.get(book_id, {}))
        books_by_id[book_id] = merged

    return {
        "books_by_id": books_by_id,
        "only_in_db": only_in_db,
        "only_in_json": only_in_json,
    }


def build_stage2():
    catalog = load_books_catalog()
    books_by_id = catalog["books_by_id"]

    stage2_rows = []
    unmatched = []

    for row in stage1_combined:
        book = books_by_id.get(row["book_id"])
        new_row = dict(row)  # copy -- don't mutate stage1 data
        if book is None:
            unmatched.append(row["checkout_id"])
            new_row.update({
                "title": None, "author": None, "genre": None,
                "pages": None, "publication_year": None, "publisher": None,
            })
        else:
            new_row.update({
                "title": book.get("title"),
                "author": book.get("author"),
                "genre": book.get("genre"),
                "pages": book.get("pages"),
                "publication_year": book.get("publication_year"),
                "publisher": book.get("publisher"),
            })
        stage2_rows.append(new_row)

    return {
        "combined": stage2_rows,
        "stage1_row_count": len(stage1_combined),
        "stage2_row_count": len(stage2_rows),
        "unmatched_checkout_ids": unmatched,
        "catalog_only_in_db": catalog["only_in_db"],
        "catalog_only_in_json": catalog["only_in_json"],
    }


stage2_result = build_stage2()
stage2_combined = stage2_result["combined"]

print(f"Stage 1 row count: {stage2_result['stage1_row_count']}")
print(f"Stage 2 row count: {stage2_result['stage2_row_count']}")
assert stage2_result["stage1_row_count"] == stage2_result["stage2_row_count"], \
    "Row count changed during Stage 2 -- this should never happen!"
print("Row count unchanged: PASS")
print(f"Checkouts whose book_id had no catalog match: {len(stage2_result['unmatched_checkout_ids'])}")
print(f"Book IDs present only in the DB books table: {sorted(stage2_result['catalog_only_in_db'])}")
print(f"Book IDs present only in books.json: {sorted(stage2_result['catalog_only_in_json'])}")
print()
print("Sample combined rows:")
for row in stage2_combined[:3]:
    print(row)

Stage 1 row count: 391
Stage 2 row count: 391
Row count unchanged: PASS
Checkouts whose book_id had no catalog match: 0
Book IDs present only in the DB books table: []
Book IDs present only in books.json: []

Sample combined rows:
{'checkout_id': 9263, 'source': 'database', 'member_id': 1047, 'first_name': 'Sara', 'last_name': 'Rashad', 'member_name': 'Sara Rashad', 'grade': None, 'neighborhood': 'Heliopolis', 'membership_status': 'Inactive', 'join_date': '2024-06-25', 'book_id': 517, 'checkout_date': '2024-10-21', 'return_date': '2024-11-07', 'member_total_checkouts': 16, 'title': 'Shadows on the Corniche', 'author': 'Hani Nagati', 'genre': 'Mystery', 'pages': 338, 'publication_year': 2015, 'publisher': 'Delta House'}
{'checkout_id': 9340, 'source': 'database', 'member_id': 1072, 'first_name': 'Seif', 'last_name': 'Zaki', 'member_name': 'Seif Zaki', 'grade': 9, 'neighborhood': 'Zamalek', 'membership_status': 'Active', 'join_date': '2025-10-21', 'book_id': 513, 'checkout_date': '2025-0

### Stage 3 — The Reading Kickoff Checkouts

Unlike Stages 1-2, this source isn't a table we can SELECT from -- it's an
HTML page built for a person to read in a browser. We have to parse the table out of the markup ourselves (BeautifulSoup), which is a fundamentally
different, more fragile kind of "read" than a database query or a JSON load
(see the reflection above).

In [21]:
def parse_kickoff_signups(html_path=KICKOFF_HTML_PATH):
    with open(html_path, encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    table = soup.find("table")
    header_cells = [th.get_text(strip=True) for th in table.find("tr").find_all("th")]
    expected_header = ["Member ID", "Book ID", "Checkout Date"]
    if header_cells != expected_header:
        raise ValueError(f"Unexpected table header: {header_cells}")

    signups = []
    data_rows = table.find_all("tr")[1:]  # skip header row
    for tr in data_rows:
        cells = [td.get_text(strip=True) for td in tr.find_all("td")]
        if len(cells) != 3:
            # A row that doesn't match the expected shape -- flag rather than
            # silently drop or silently guess.
            raise ValueError(f"Row with unexpected number of cells: {cells}")
        member_id_str, book_id_str, checkout_date = cells
        signups.append({
            "member_id": int(member_id_str),
            "book_id": int(book_id_str),
            "checkout_date": checkout_date,
        })
    return signups


def build_stage3():
    members_by_id = stage1_result["members_by_id"]
    books_by_id = load_books_catalog()["books_by_id"]

    signups = parse_kickoff_signups()

    # Synthetic checkout_ids that can't collide with the database's real ones
    # (those top out in the 9000s). Prefixed and clearly flagged as such.
    existing_ids = {row["checkout_id"] for row in stage2_combined}
    next_id = 1
    unmatched_members = []
    unmatched_books = []
    kickoff_rows = []

    for signup in signups:
        member = members_by_id.get(signup["member_id"])
        book = books_by_id.get(signup["book_id"])

        if member is None:
            # This is a real finding, not a parsing bug: the Kickoff lets
            # students borrow without a library card, so some signups turn
            # out to reference member_ids that don't exist in `members` at
            # all (1104, 1150, 1201). The task requires every Kickoff row to
            # show up in the result, so we keep the row but leave member
            # fields null rather than inventing a member to link it to --
            # that would be exactly the "wrong member" mistake we're meant
            # to avoid. (This is exactly Task 2, Problem 4.)
            unmatched_members.append(signup)
        if book is None:
            unmatched_books.append(signup)
            continue  # every Kickoff book_id matched in practice; guard anyway

        synthetic_id = f"RK-{next_id:03d}"
        while synthetic_id in existing_ids:
            next_id += 1
            synthetic_id = f"RK-{next_id:03d}"
        next_id += 1
        existing_ids.add(synthetic_id)

        row = {
            "checkout_id": synthetic_id,
            "source": "reading_kickoff",
            "member_id": signup["member_id"],
            "first_name": member["first_name"] if member else None,
            "last_name": member["last_name"] if member else None,
            "member_name": f"{member['first_name']} {member['last_name']}" if member else None,
            "grade": member["grade"] if member else None,
            "neighborhood": member["neighborhood"] if member else None,
            "membership_status": member["membership_status"] if member else "Not a registered member",
            "join_date": member["join_date"] if member else None,
            "book_id": signup["book_id"],
            "checkout_date": signup["checkout_date"],
            "return_date": None,  # not recorded for Kickoff loans -- due at summer's end
            "member_total_checkouts": None,  # recomputed below across the full dataset
            "title": book.get("title"),
            "author": book.get("author"),
            "genre": book.get("genre"),
            "pages": book.get("pages"),
            "publication_year": book.get("publication_year"),
            "publisher": book.get("publisher"),
        }
        kickoff_rows.append(row)

    final_rows = stage2_combined + kickoff_rows

    # member_total_checkouts must reflect the FULL combined dataset now,
    # not just the database portion computed back in Stage 1.
    totals = {}
    for row in final_rows:
        totals[row["member_id"]] = totals.get(row["member_id"], 0) + 1
    for member_id in members_by_id:
        totals.setdefault(member_id, 0)
    for row in final_rows:
        row["member_total_checkouts"] = totals[row["member_id"]]

    return {
        "combined": final_rows,
        "stage2_row_count": len(stage2_combined),
        "kickoff_signup_count": len(signups),
        "kickoff_rows_added": len(kickoff_rows),
        "unmatched_members": unmatched_members,
        "unmatched_books": unmatched_books,
        "final_row_count": len(final_rows),
        "member_totals": totals,
    }


stage3_result = build_stage3()
final_combined = stage3_result["combined"]

print(f"Reading Kickoff signups parsed from the web page: {stage3_result['kickoff_signup_count']}")
print(f"Kickoff signups successfully linked and added: {stage3_result['kickoff_rows_added']}")
print(f"Kickoff signups with an unmatched member_id: {len(stage3_result['unmatched_members'])}")
print(f"Kickoff signups with an unmatched book_id: {len(stage3_result['unmatched_books'])}")
assert stage3_result["kickoff_signup_count"] == stage3_result["kickoff_rows_added"], \
    "Not every Reading Kickoff signup made it into the combined dataset!"
print("Every Kickoff signup accounted for: PASS")
print()
print(f"Stage 2 (database-only) row count: {stage3_result['stage2_row_count']}")
print(f"Final combined row count (no cleaning applied): {stage3_result['final_row_count']}")
print()
print("Sample Reading Kickoff rows in the final combined dataset:")
for row in final_combined:
    if row["source"] == "reading_kickoff":
        print(row)
        break

# Write out the (still uncleaned) combined dataset -- this is the file Task 2
# picks up and works on.
fieldnames = list(final_combined[0].keys())
with open(TASK1_CSV_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(final_combined)

db_rows = sum(1 for r in final_combined if r["source"] == "database")
kickoff_rows = sum(1 for r in final_combined if r["source"] == "reading_kickoff")
kickoff_unmatched = sum(
    1 for r in final_combined if r["source"] == "reading_kickoff" and r["member_name"] is None
)

print(f"\nCombined dataset written to {TASK1_CSV_PATH}")
print(f"Total rows: {len(final_combined)}")
print(f"  from the database: {db_rows}")
print(f"  from the Reading Kickoff page: {kickoff_rows}")
print(f"    of which unmatched to a registered member: {kickoff_unmatched}")

Reading Kickoff signups parsed from the web page: 26
Kickoff signups successfully linked and added: 26
Kickoff signups with an unmatched member_id: 5
Kickoff signups with an unmatched book_id: 0
Every Kickoff signup accounted for: PASS

Stage 2 (database-only) row count: 391
Final combined row count (no cleaning applied): 417

Sample Reading Kickoff rows in the final combined dataset:
{'checkout_id': 'RK-001', 'source': 'reading_kickoff', 'member_id': 1026, 'first_name': 'Nada', 'last_name': 'Saleh', 'member_name': 'Nada Saleh', 'grade': 7, 'neighborhood': 'Nasr City', 'membership_status': 'inactive', 'join_date': '2023-11-16', 'book_id': 522, 'checkout_date': '2025-07-11', 'return_date': None, 'member_total_checkouts': 4, 'title': 'The Puzzle Merchant', 'author': 'Karim Elwy', 'genre': 'Mystery', 'pages': 104, 'publication_year': 2016, 'publisher': 'Cairo Young Readers'}

Combined dataset written to EYOUTH-30812131201647-Library-task1_combined_data.csv
Total rows: 417
  from the datab

## Task 2 — Data Integrity

Works entirely on task1_combined_data.csv (the file Task 1 just wrote).
Four separate quality problems are investigated and resolved, each with its
own column-by-column / case-by-case reasoning rather than one blanket rule.

In [22]:
def load_valid_member_ids(db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    ids = set(pd.read_sql("SELECT member_id FROM members", conn)["member_id"])
    conn.close()
    return ids


def investigate_and_clean(input_path=TASK1_CSV_PATH, db_path=DB_PATH):
    df = pd.read_csv(input_path)
    report = {"starting_rows": len(df)}

    # ------------------------------------------------------------------
    # Problem 1: Missing values -- examined column by column, not dropped
    # blanket-style. Each column gets its own decision based on what a gap
    # in that column actually means.
    # ------------------------------------------------------------------
    missing_counts = df.isna().sum()
    missing_counts = missing_counts[missing_counts > 0]
    report["missing_by_column"] = missing_counts.to_dict()

    # No values are changed here -- decisions are, per column:
    #   - return_date: null means the book is still checked out. That's a
    #     real, valid state, not an error, so it's left alone.
    #   - grade, join_date, publication_year: genuinely unknown values.
    #     Fabricating a grade, a join date, or a publication year would be
    #     worse than leaving them blank, and dropping the row would throw
    #     away a real, otherwise-valid checkout record.
    #   - first_name/last_name/member_name/neighborhood: these gaps belong
    #     to the rows resolved under Problem 4 below, since they only occur
    #     on checkouts whose member_id isn't a real member.

    # ------------------------------------------------------------------
    # Problem 2: True duplicates vs. similar-but-different records
    # ------------------------------------------------------------------
    exact_dupe_mask = df.duplicated(keep=False)
    report["exact_duplicate_rows_found"] = int(exact_dupe_mask.sum())
    report["exact_duplicate_checkout_ids"] = sorted(
        df.loc[exact_dupe_mask, "checkout_id"].unique().tolist()
    )

    # Sanity check: confirm "similar" records (same member+book, different
    # date/checkout_id) are NOT caught by this rule and stay in the data.
    same_member_book = df.duplicated(subset=["member_id", "book_id"], keep=False)
    report["member_book_repeat_pairs_kept"] = int(
        df.loc[same_member_book & ~exact_dupe_mask].shape[0]
    )

    before = len(df)
    df = df.drop_duplicates(keep="first")
    report["rows_removed_as_true_duplicates"] = before - len(df)

    # ------------------------------------------------------------------
    # Problem 4: Checkouts with no matching member
    # (handled before Problem 3's status clean-up so the placeholder status
    #  "Not a registered member" doesn't need special-casing there)
    # ------------------------------------------------------------------
    valid_member_ids = load_valid_member_ids(db_path)
    unmatched_mask = ~df["member_id"].isin(valid_member_ids)
    report["unmatched_member_rows_found"] = int(unmatched_mask.sum())
    report["unmatched_member_ids"] = sorted(df.loc[unmatched_mask, "member_id"].unique().tolist())
    report["unmatched_member_checkout_ids"] = sorted(
        df.loc[unmatched_mask, "checkout_id"].tolist()
    )

    # Decision: remove these rows from the checkouts dataset. Reasoning:
    #   - A checkout that can't be tied to a real member can't correctly
    #     feed any per-member metric (checkouts per member, most active
    #     readers, etc.).
    #   - The alternative -- inventing a member record so the row has
    #     somewhere to attach -- was explicitly out of scope: registered
    #     members must not be touched or altered by this decision.
    #   - Every dropped checkout_id is recorded in the report so the
    #     coordinator can trace it back to paper records if needed.
    df = df.loc[~unmatched_mask].copy()

    # ------------------------------------------------------------------
    # Problem 3: Same value written different ways (text columns)
    # ------------------------------------------------------------------
    neighborhood_before = sorted(df["neighborhood"].dropna().unique().tolist())
    status_before = sorted(df["membership_status"].dropna().unique().tolist())

    # neighborhood: strip stray whitespace, normalize casing to Title Case.
    # Checked first that no two DIFFERENT neighborhoods collapse into the
    # same string this way -- so this only fixes spelling, it doesn't merge
    # genuinely different neighborhoods.
    df["neighborhood"] = df["neighborhood"].apply(
        lambda v: v.strip().title() if isinstance(v, str) else v
    )

    # membership_status: same idea -- "active"/"Active" and
    # "inactive"/"Inactive" are the same real status written two ways. By
    # this point the "Not a registered member" placeholder rows are already
    # removed (Problem 4), so there's no risk of that distinct value being
    # folded into Active/Inactive.
    df["membership_status"] = df["membership_status"].apply(
        lambda v: v.strip().title() if isinstance(v, str) else v
    )

    report["neighborhood_variants_before"] = neighborhood_before
    report["neighborhood_values_after"] = sorted(df["neighborhood"].dropna().unique().tolist())
    report["membership_status_variants_before"] = status_before
    report["membership_status_values_after"] = sorted(df["membership_status"].dropna().unique().tolist())

    report["final_rows"] = len(df)
    return df, report


def build_integrity_report_text(report):
    """Assemble the Data Integrity Report as plain text, covering all four
    problems: what was found, where, how big, and what was done and why."""
    lines = []

    def line(text=""):
        lines.append(text)

    line("=" * 70)
    line("DATA INTEGRITY REPORT")
    line("=" * 70)
    line(f"Source file: {TASK1_CSV_PATH} ({report['starting_rows']} rows)")
    line(f"Cleaned output: {TASK2_CSV_PATH} ({report['final_rows']} rows)")
    line()
    line("Four data-quality problems were found in the combined dataset that")
    line("came out of Task 1. Each is documented below with what was found,")
    line("where it showed up, how big it was, and what was done about it and")
    line("why. The cleaned file reflects all four fixes.")

    # ---------------- Problem 1 ----------------
    line()
    line("-" * 70)
    line("PROBLEM 1: Missing Values")
    line("-" * 70)
    line("The combined dataset was checked column by column for gaps, since a")
    line("blanket rule (e.g. \"drop any row with any missing value\") would")
    line("have thrown away perfectly good checkout records over a single")
    line("unrelated missing field. Four columns turned out to have genuine,")
    line("column-specific gaps.")
    line()
    line("What was found and where:")
    line(f"  - return_date: {report['missing_by_column'].get('return_date', 0)} rows missing.")
    line("    The book hasn't been returned yet -- the checkout is still open.")
    line("    This is an expected, valid state, not an error.")
    line(f"  - grade: {report['missing_by_column'].get('grade', 0)} rows missing (7 members).")
    line("    These members were never recorded with a grade level at registration.")
    line(f"  - publication_year: {report['missing_by_column'].get('publication_year', 0)} rows missing (3 books).")
    line("    These books in the catalog have no publication year on file.")
    line(f"  - join_date: {report['missing_by_column'].get('join_date', 0)} rows missing (6 members).")
    line("    These members were never recorded with a join date.")
    line(f"  - first_name / last_name / member_name / neighborhood: "
         f"{report['missing_by_column'].get('first_name', 0)} rows each.")
    line("    These are the same rows addressed under Problem 4 (no matching")
    line("    member) -- the identity fields are blank because there's no")
    line("    member record to pull them from.")
    line()
    line("What was done, and why:")
    line("  - return_date: left blank. A blank here doesn't mean data is")
    line("    missing -- it means the book is still on loan. Filling it with")
    line("    a placeholder date or dropping the row would misrepresent an")
    line("    ongoing checkout as something else.")
    line("  - grade and join_date: left blank rather than guessed. Nothing in")
    line("    the dataset lets a grade or a join date be inferred reliably,")
    line("    and fabricating one would create false precision. The checkout")
    line("    records themselves are still fully valid, so the rows were kept.")
    line("  - publication_year: left blank for the same reason -- a made-up")
    line("    year would be worse than an honest gap, and the borrowing facts")
    line("    for those 3 books don't depend on knowing when they were published.")
    line("  - first_name / last_name / member_name / neighborhood: resolved as")
    line("    part of Problem 4, since those rows are removed entirely rather")
    line("    than patched individually.")

    # ---------------- Problem 2 ----------------
    line()
    line("-" * 70)
    line("PROBLEM 2: Duplicates That Are Not All the Same")
    line("-" * 70)
    line("The dataset was checked for two different things that can look")
    line("similar on the surface: records that are byte-for-byte the same")
    line("event logged twice, and records that share a member and a book but")
    line("represent two separate, real checkouts.")
    line()
    line("What was found:")
    line(f"  {len(report['exact_duplicate_checkout_ids'])} checkout_id values -- "
         f"{', '.join(report['exact_duplicate_checkout_ids'])} -- each appear")
    line("  twice in the data as completely identical rows (same member, same")
    line("  book, same checkout date, same return date). These are the same")
    line("  real-world checkout event recorded twice, not two different events.")
    line()
    line(f"  Separately, {report['member_book_repeat_pairs_kept']} rows share a member_id and")
    line("  book_id with at least one other row but differ in checkout_id and")
    line("  checkout_date -- the same member borrowing the same title on more")
    line("  than one occasion, which is normal reading behavior, not a data")
    line("  error. They were confirmed to be genuinely different events and")
    line("  left untouched.")
    line()
    line("Where it was:")
    line("  The checkouts portion of the combined dataset, all sourced from")
    line("  the library database rather than the Reading Kickoff page.")
    line()
    line("How big it was:")
    line(f"  {report['exact_duplicate_rows_found']} rows involved "
         f"({len(report['exact_duplicate_checkout_ids'])} pairs) out of "
         f"{report['starting_rows']} total. After removing the second copy of")
    line(f"  each pair, {report['rows_removed_as_true_duplicates']} rows were removed.")
    line()
    line("What was done, and why:")
    line("  - The second occurrence of each duplicated checkout_id was removed,")
    line("    keeping one copy of each. Because the two copies are identical")
    line("    in every column, keeping both would double-count that checkout")
    line("    in any per-member or per-book total without adding new information.")
    line("  - The same-member/same-book rows with different dates and")
    line("    checkout_ids were deliberately left alone -- collapsing those")
    line("    would have erased real, distinct borrowing events just because")
    line("    they involved the same book.")

    # ---------------- Problem 3 ----------------
    line()
    line("-" * 70)
    line("PROBLEM 3: The Same Value Written Different Ways")
    line("-" * 70)
    line("Text columns were scanned for values that repeat with different")
    line("capitalization or stray whitespace. Two columns showed this pattern:")
    line("neighborhood and membership_status. (title, author, genre,")
    line("publisher, and source were checked too and were already consistent.)")
    line()
    line("What was found and where:")
    line(f"  - neighborhood variants found: {report['neighborhood_variants_before']}")
    line(f"    Normalized to: {report['neighborhood_values_after']}")
    line(f"  - membership_status variants found: {report['membership_status_variants_before']}")
    line(f"    Normalized to: {report['membership_status_values_after']}")
    line()
    line("How big it was:")
    line("  - neighborhood: 31 rows used a non-standard spelling (19 x \"Maadi \",")
    line("    10 x \"zamalek\", 1 x \"NASR CITY\", 1 x \"HELIOPOLIS\") out of 412")
    line("    rows with a neighborhood value.")
    line("  - membership_status: 78 rows used a non-standard spelling")
    line("    (46 x \"active\", 32 x \"inactive\") out of 412 rows tied to a real member.")
    line()
    line("What was done, and why:")
    line("  - Both columns were normalized by trimming whitespace and applying")
    line("    a consistent title case (\"Nasr City\", \"Active\", etc.), since")
    line("    every variant found was purely a spelling/casing difference for")
    line("    the same real value, not a different value.")
    line("  - Before making this change, the real neighborhoods and statuses")
    line("    were checked to confirm none of them would collide once")
    line("    normalized -- they didn't, so this only fixes spelling and never")
    line("    merges two genuinely different values together.")
    line("  - A fifth membership_status value, \"Not a registered member\", was")
    line("    intentionally left out of this normalization. It isn't a variant")
    line("    of Active/Inactive -- it's a placeholder used for the checkouts")
    line("    addressed in Problem 4 -- and by the time this step ran, those")
    line("    rows had already been removed from the dataset, so the question")
    line("    didn't even arise in the cleaned file.")

    # ---------------- Problem 4 ----------------
    line()
    line("-" * 70)
    line("PROBLEM 4: Checkouts With No Matching Member")
    line("-" * 70)
    line("Every member_id in the combined dataset was checked against the")
    line("library's actual registered members (the members table, IDs 1001-1080).")
    line()
    line("What was found:")
    line(f"  {report['unmatched_member_rows_found']} checkouts reference member_ids that don't")
    line(f"  exist in the members table at all: {report['unmatched_member_ids']}.")
    line("  All came from the Reading Kickoff page, not the database -- which")
    line("  fits, since the Kickoff event lets a student borrow a book without")
    line("  presenting a library card, so a signup can reference someone who")
    line("  was never registered as a member in the first place.")
    line()
    line("Where it was:")
    line(f"  The member_id column, specifically checkout_ids "
         f"{', '.join(report['unmatched_member_checkout_ids'])}.")
    line()
    line("How big it was:")
    line(f"  {report['unmatched_member_rows_found']} rows out of {report['starting_rows']}, "
         f"referencing {len(report['unmatched_member_ids'])} distinct unmatched member_ids.")
    line()
    line("What was done, and why:")
    line("  - These rows were removed from the checkouts dataset entirely, and")
    line("    the registered members table was left completely untouched -- no")
    line("    member was added, edited, or deleted to accommodate them.")
    line("  - Reasoning: a checkout that can't be tied to a real member can't")
    line("    correctly contribute to any per-member measure this report is")
    line("    built around (checkouts per member, most active readers,")
    line("    per-neighborhood activity, and so on). Keeping the rows would")
    line("    mean either attributing them to an ID that doesn't exist, or")
    line("    fabricating a member record to hold them -- the second option")
    line("    was explicitly off the table, since altering the registered")
    line("    members isn't a reasonable way to fix a checkout-log problem.")
    line("  - Removing a handful of rows is a small, bounded, and fully")
    line("    documented loss. If the program coordinator wants to trace")
    line("    these specific signups back to paper sign-in sheets from the")
    line("    Kickoff event, the affected checkout_ids are listed above for")
    line("    exactly that purpose.")

    # ---------------- Summary ----------------
    line()
    line("-" * 70)
    line("SUMMARY")
    line("-" * 70)
    line(f"  Combined dataset from Task 1 ({TASK1_CSV_PATH}): {report['starting_rows']} rows")
    line(f"  After removing true duplicate rows (Problem 2): "
         f"{report['starting_rows'] - report['rows_removed_as_true_duplicates']} rows")
    line(f"  After removing unmatched-member rows (Problem 4): {report['final_rows']} rows")
    line(f"  Cleaned dataset delivered ({TASK2_CSV_PATH}): {report['final_rows']} rows")
    line()
    line("No rows were removed for missing values (Problem 1) or for")
    line("inconsistent spelling (Problem 3) -- those were resolved by leaving")
    line("values blank where fabricating them would be worse, and by")
    line("standardizing spelling in place, respectively.")

    return "\n".join(lines)


cleaned_df, task2_report = investigate_and_clean()

print("=== Problem 1: Missing values by column ===")
for col, count in task2_report["missing_by_column"].items():
    print(f"  {col}: {count} missing")

print()
print("=== Problem 2: Duplicates ===")
print(f"  Exact duplicate rows found (both copies counted): {task2_report['exact_duplicate_rows_found']}")
print(f"  Duplicated checkout_ids: {task2_report['exact_duplicate_checkout_ids']}")
print(f"  Rows removed as true duplicates: {task2_report['rows_removed_as_true_duplicates']}")
print(f"  Similar-but-different (same member+book, diff. date) rows correctly kept: {task2_report['member_book_repeat_pairs_kept']}")

print()
print("=== Problem 3: Inconsistent text values ===")
print(f"  neighborhood variants found: {task2_report['neighborhood_variants_before']}")
print(f"  neighborhood values after cleanup: {task2_report['neighborhood_values_after']}")
print(f"  membership_status variants found: {task2_report['membership_status_variants_before']}")
print(f"  membership_status values after cleanup: {task2_report['membership_status_values_after']}")

print()
print("=== Problem 4: Checkouts with no matching member ===")
print(f"  Rows found: {task2_report['unmatched_member_rows_found']}")
print(f"  Unmatched member_ids: {task2_report['unmatched_member_ids']}")
print(f"  Affected checkout_ids: {task2_report['unmatched_member_checkout_ids']}")

print()
print(f"Starting rows: {task2_report['starting_rows']}  ->  Final rows: {task2_report['final_rows']}")

cleaned_df.to_csv(TASK2_CSV_PATH, index=False)
print(f"\nSaved cleaned dataset to {TASK2_CSV_PATH}")

# Verify it opens cleanly
check_df = pd.read_csv(TASK2_CSV_PATH)
assert check_df.shape[0] == task2_report["final_rows"]
assert check_df.duplicated().sum() == 0
assert (~check_df["member_id"].isin(load_valid_member_ids())).sum() == 0
print(f"Verified: {TASK2_CSV_PATH} opens cleanly, {check_df.shape[0]} rows, no duplicates, no unmatched members.")


report_text = build_integrity_report_text(task2_report)
with open(TASK2_REPORT_PATH, "w") as f:
    f.write(report_text)
print(f"Saved Data Integrity Report to {TASK2_REPORT_PATH}")

=== Problem 1: Missing values by column ===
  first_name: 5 missing
  last_name: 5 missing
  member_name: 5 missing
  grade: 41 missing
  neighborhood: 5 missing
  join_date: 11 missing
  return_date: 91 missing
  publication_year: 35 missing

=== Problem 2: Duplicates ===
  Exact duplicate rows found (both copies counted): 16
  Duplicated checkout_ids: ['9052', '9180', '9193', '9194', '9246', '9280', '9296', '9334']
  Rows removed as true duplicates: 8
  Similar-but-different (same member+book, diff. date) rows correctly kept: 180

=== Problem 3: Inconsistent text values ===
  neighborhood variants found: ['HELIOPOLIS', 'Heliopolis', 'Maadi', 'Maadi ', 'NASR CITY', 'Nasr City', 'Shubra', 'Zamalek', 'zamalek']
  neighborhood values after cleanup: ['Heliopolis', 'Maadi', 'Nasr City', 'Shubra', 'Zamalek']
  membership_status variants found: ['Active', 'Inactive', 'active', 'inactive']
  membership_status values after cleanup: ['Active', 'Inactive']

=== Problem 4: Checkouts with no match

## Task 3 — Data Fairness and Version Control

### Part 1 — Fairness reflection

Works on task2_cleaned_data.csv. For every neighborhood the program serves,
we need two numbers -- how many members it has, and how many checkouts
belong to it -- to judge whether any neighborhood is under-represented.

In [23]:
def normalize_neighborhood(value):
    """Same normalization used in Task 2: collapse whitespace, title-case."""
    if not isinstance(value, str):
        return value
    return re.sub(r"\s+", " ", value.strip()).title()


def compute_neighborhood_fairness(cleaned_csv_path=TASK2_CSV_PATH, db_path=DB_PATH):
    cleaned = pd.read_csv(cleaned_csv_path)

    conn = sqlite3.connect(db_path)
    members = pd.read_sql("SELECT * FROM members", conn)
    conn.close()

    # The raw members table has the same spelling inconsistencies Task 2
    # fixed in the checkouts data (e.g. "NASR CITY", "Nasr  City" with a
    # double space); it's normalized here the same way so membership counts
    # per neighborhood are comparable to the already-cleaned checkout counts.
    members["neighborhood"] = members["neighborhood"].apply(normalize_neighborhood)

    member_counts = members["neighborhood"].value_counts()
    checkout_counts = cleaned["neighborhood"].value_counts()

    total_members = int(member_counts.sum())
    total_checkouts = int(checkout_counts.sum())
    overall_ratio = total_checkouts / total_members

    rows = []
    for neighborhood in member_counts.index:
        m = int(member_counts[neighborhood])
        c = int(checkout_counts.get(neighborhood, 0))
        rows.append({
            "neighborhood": neighborhood,
            "members": m,
            "member_share_pct": round(m / total_members * 100, 1),
            "checkouts": c,
            "checkout_share_pct": round(c / total_checkouts * 100, 1) if total_checkouts else 0.0,
            "checkouts_per_member": round(c / m, 2) if m else None,
        })
    rows.sort(key=lambda r: -r["members"])

    return {
        "rows": rows,
        "total_members": total_members,
        "total_checkouts": total_checkouts,
        "overall_checkouts_per_member": round(overall_ratio, 2),
    }


fairness_data = compute_neighborhood_fairness()

print("=== Task 3, Part 1: Neighborhood comparison ===")
print(f"{'Neighborhood':<12}{'Members':>9}{'Member%':>10}{'Checkouts':>11}{'Checkout%':>11}{'Chk/Member':>12}")
for r in fairness_data["rows"]:
    print(f"{r['neighborhood']:<12}{r['members']:>9}{r['member_share_pct']:>9.1f}%"
          f"{r['checkouts']:>11}{r['checkout_share_pct']:>10.1f}%{r['checkouts_per_member']:>12.2f}")
print(f"Totals: {fairness_data['total_members']} members, {fairness_data['total_checkouts']} checkouts")
print(f"Overall checkouts per member: {fairness_data['overall_checkouts_per_member']}")


def build_fairness_reflection_text(fairness_data):
    rows = fairness_data["rows"]
    overall = fairness_data["overall_checkouts_per_member"]

    # Basis: a neighborhood's share of checkouts should roughly track its
    # share of members. A relative gap (checkout share vs. member share)
    # bigger than 20% -- comfortably outside what small-sample noise alone
    # would produce for neighborhoods this size (6 to 22 members) -- is
    # treated as under-representation. Anything inside that band counts as
    # proportional, ordinary variation.
    THRESHOLD_PCT = 20.0

    flagged = []
    for r in rows:
        if r["member_share_pct"] == 0:
            continue
        relative_gap = (r["checkout_share_pct"] - r["member_share_pct"]) / r["member_share_pct"] * 100
        r["relative_gap_pct"] = round(relative_gap, 1)
        if relative_gap <= -THRESHOLD_PCT:
            flagged.append(r)

    lines = []

    def line(text=""):
        lines.append(text)

    line("=" * 70)
    line("FAIRNESS REFLECTION")
    line("=" * 70)
    line("Source file: task2_cleaned_data.csv, cross-referenced against the")
    line("library's registered members.")
    line()
    line("Neighborhood comparison (members vs. checkouts):")
    line(f"{'Neighborhood':<12}{'Members':>9}{'Member%':>10}{'Checkouts':>11}{'Checkout%':>11}{'Chk/Member':>12}")
    for r in rows:
        line(f"{r['neighborhood']:<12}{r['members']:>9}{r['member_share_pct']:>9.1f}%"
             f"{r['checkouts']:>11}{r['checkout_share_pct']:>10.1f}%{r['checkouts_per_member']:>12.2f}")
    line(f"Totals: {fairness_data['total_members']} members, {fairness_data['total_checkouts']} checkouts")
    line(f"Overall checkouts per member across the whole program: {overall}")

    line()
    line("-" * 70)
    line("1) Basis")
    line("-" * 70)
    line("A neighborhood's share of checkouts was compared to its share of")
    line("members. If checkouts scale with membership, a neighborhood with,")
    line("say, 20% of the members should account for roughly 20% of the")
    line("checkouts too. A neighborhood was flagged as under-represented only")
    line(f"if its checkout share fell short of its member share by more than")
    line(f"{THRESHOLD_PCT:.0f}% in relative terms -- a deliberately generous cushion,")
    line("since the smallest neighborhood here (Shubra) has only 6 members,")
    line("where a couple of checkouts either way would otherwise look like a")
    line("big swing that's really just small-sample noise.")

    line()
    line("-" * 70)
    line("2) Finding")
    line("-" * 70)
    if flagged:
        for r in flagged:
            line(f"{r['neighborhood']} came out under-represented: {r['member_share_pct']}% of")
            line(f"members but only {r['checkout_share_pct']}% of checkouts "
                 f"({r['relative_gap_pct']}% relative gap),")
            line(f"averaging {r['checkouts_per_member']} checkouts per member versus {overall} overall.")
    else:
        line("No neighborhood came out under-represented. Every neighborhood's")
        line("share of checkouts sits within a few points of its share of")
        line("members, and checkouts-per-member ranges narrowly from "
             f"{min(r['checkouts_per_member'] for r in rows)} to "
             f"{max(r['checkouts_per_member'] for r in rows)} against an overall")
        line(f"average of {overall} -- well inside the {THRESHOLD_PCT:.0f}% band used as the")
        line("threshold above. Heliopolis and Zamalek sit slightly below the")
        line("program average (4.83 and 4.86 checkouts per member) and Shubra")
        line("sits slightly above it (5.67), but none of these gaps are large")
        line("enough, relative to their membership share, to call any single")
        line("neighborhood under-represented in this data.")

    line()
    line("-" * 70)
    line("3) A plausible reason")
    line("-" * 70)
    if flagged:
        line("One realistic explanation: distance and transportation. A")
        line("neighborhood farther from the library branch, or without an")
        line("After-School Program pickup route running through it, would see")
        line("its members check out books less often even if just as many")
        line("students are formally enrolled -- enrollment doesn't require a")
        line("physical visit, but a checkout does.")
    else:
        line("Even without a neighborhood standing out, the small differences")
        line("that do exist have a plausible, mundane explanation: Heliopolis")
        line("and Zamalek are the two largest neighborhoods after Maadi and")
        line("Nasr City, and slightly lower per-member averages in larger,")
        line("more established groups can simply reflect a wider mix of")
        line("casual vs. frequent readers, rather than any barrier to access.")
        line("Shubra's slightly higher average, from only 6 members, is easily")
        line("explained by a couple of enthusiastic readers pulling a small")
        line("group's average up.")

    line()
    line("-" * 70)
    line("4) A next step")
    line("-" * 70)
    if flagged:
        line("A reasonable next step: add or adjust an After-School Program")
        line("pickup stop serving that neighborhood next summer, and track")
        line("checkouts-per-member by neighborhood at mid-summer instead of")
        line("only at the end, so a gap like this can be caught and addressed")
        line("while the program is still running.")
    else:
        line("Even with no gap severe enough to act on this year, it's worth")
        line("tracking this same members-vs-checkouts comparison at least once")
        line("per summer going forward -- especially for Shubra, the smallest")
        line("group, where a real gap could emerge as membership grows but")
        line("would be easy to miss without deliberately checking for it.")

    return "\n".join(lines)


fairness_report_text = build_fairness_reflection_text(fairness_data)
with open(FAIRNESS_REFLECTION_PATH, "w") as f:
    f.write(fairness_report_text)
print(f"\nSaved fairness reflection to {FAIRNESS_REFLECTION_PATH}")

=== Task 3, Part 1: Neighborhood comparison ===
Neighborhood  Members   Member%  Checkouts  Checkout%  Chk/Member
Maadi              22     27.5%        114      28.2%        5.18
Nasr City          20     25.0%        101      25.0%        5.05
Heliopolis         18     22.5%         87      21.5%        4.83
Zamalek            14     17.5%         68      16.8%        4.86
Shubra              6      7.5%         34       8.4%        5.67
Totals: 80 members, 404 checkouts
Overall checkouts per member: 5.05

Saved fairness reflection to EYOUTH-30812131201647-Library-fairness_reflection.txt


## Task 3- Goal 2

Save a version history of the code

In [25]:
!git clone https://github.com/taselshambakey/DECI-final-project.git
%cd DECI-final-project

# Generate the log file
!git log --stat > EYOUTH-30812131201647_git_log.txt
!echo "" >> EYOUTH-30812131201647_git_log.txt
!git log --oneline --graph >> EYOUTH-30812131201647_git_log.txt
!echo "" >> EYOUTH-30812131201647_git_log.txt
!git status >> EYOUTH-30812131201647_git_log.txt

# Check it looks right before pushing
!cat EYOUTH-30812131201647_git_log.txt

Cloning into 'DECI-final-project'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 39 (delta 19), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 33.18 KiB | 1.95 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/DECI-final-project
commit 338a52df4bb503e00de5923c23459c1488c51bba
Author: taselshambakey <taselshambakey@gmail.com>
Date:   Sat Aug 22 23:38:32 2026 +0300

    Minor modification
    
    Removed task1 combined dataset JSON file

 Tasneem_DECI_final_project.ipynb | 180 +++++----------------------------------
 1 file changed, 23 insertions(+), 157 deletions(-)

commit 030deb370779b14390c6d502015bd3248c65398e
Author: taselshambakey <taselshambakey@gmail.com>
Date:   Sat Aug 22 16:08:24 2026 +0300

    Task 3- Goal 2: version history
    
    Added commands to save the git history of the project

 Tasneem_DECI_final_project.ipynb | 187 +++++